In [28]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
import seaborn as sns

In [29]:
from model import IntentClassifier, BANKING77_CONFIG
from training import BANKING77_LABELS,TRAIN_CONFIG,EmbeddingDataset,train,compute_ece, ID2LABEL

In [30]:
def evaluate_on_test(model, test_loader, criterion, device, class_names=None):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []
    total_loss = 0.0

    with torch.no_grad():
        for emb, mask, labels in test_loader:  # matches your DataLoader format
            emb, mask, labels = emb.to(device), mask.to(device), labels.to(device)

            logits = model(emb, mask)  # matches your model's forward signature
            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    avg_loss = total_loss / len(test_loader)

    # --- Metrics ---
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')

    print(f"\n{'='*50}")
    print(f"  TEST RESULTS")
    print(f"{'='*50}")
    print(f"  Loss         : {avg_loss:.4f}")
    print(f"  Accuracy     : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  F1 (macro)   : {f1_macro:.4f}")
    print(f"  F1 (weighted): {f1_weighted:.4f}")
    print(f"{'='*50}\n")

    # --- Per-class report ---
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    return {
        'loss': avg_loss, 'accuracy': acc,
        'f1_macro': f1_macro, 'f1_weighted': f1_weighted,
        'preds': all_preds, 'labels': all_labels, 'probs': all_probs
    }
    


In [31]:
if __name__ == "__main__":
    test_ds = EmbeddingDataset(TRAIN_CONFIG["test_emb"])
    test_loader = DataLoader(
        test_ds,
        batch_size=TRAIN_CONFIG["batch_size"] * 2,
        shuffle=False,
        num_workers=0,
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = IntentClassifier(
        cfg=BANKING77_CONFIG,
        num_classes=TRAIN_CONFIG["num_classes"],
        pool=TRAIN_CONFIG["pool"],
    ).to(device)
    
    state_dict = torch.load('best_model.pt', weights_only=True)
    model.load_state_dict(state_dict)

    criterion = torch.nn.CrossEntropyLoss()

    # Your class names (optional but recommended)
    class_names = class_names=[ID2LABEL[i] for i in range(len(ID2LABEL) - 1)]

    results = evaluate_on_test(model, test_loader, criterion, device, class_names)


  TEST RESULTS
  Loss         : 0.3385
  Accuracy     : 0.9138 (91.38%)
  F1 (macro)   : 0.9136
  F1 (weighted): 0.9136

Classification Report:
                                                  precision    recall  f1-score   support

                                activate_my_card       0.97      0.95      0.96        40
                                       age_limit       0.98      1.00      0.99        40
                         apple_pay_or_google_pay       1.00      1.00      1.00        40
                                     atm_support       1.00      1.00      1.00        39
                                automatic_top_up       0.95      0.88      0.91        40
         balance_not_updated_after_bank_transfer       0.91      0.72      0.81        40
balance_not_updated_after_cheque_or_cash_deposit       0.97      0.97      0.97        40
                         beneficiary_not_allowed       0.90      0.90      0.90        40
                                 cancel_tran